## Seq2Seq LSTM + Teacher Forcing + Embedding + Attention

글자 알파벳 문자열을 입력받아 거꾸로 출력하는 예제입니다.

적용 개념: Seq2Seq, Teacher Forcing, Embedding, Attention\n

In [ ]:
# ============================================================
# Seq2Seq LSTM + Teacher Forcing + Embedding + Attention 적용 버전
# ------------------------------------------------------------
# 목적:
#   알파벳 대소문자로 구성된 문자열을 입력받아
#   그 문자열을 거꾸로 뒤집어 출력하는 모델을 학습한다.
#
# 예)
#   입력: AbCdE
#   출력: EdCbA
#
# 현재 설정:
#   cal_len = 52
#   즉, 입력 문자열은 반드시 52글자여야 한다.
#
# 이 코드에 포함된 핵심 개념:
#   1. Seq2Seq
#   2. Teacher Forcing
#   3. Embedding
#   4. LSTM Encoder-Decoder
#   5. Attention
# ============================================================


import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention, Concatenate
from tensorflow.keras.callbacks import EarlyStopping


# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------

# 하나의 입력 문자열 길이
# cal_len = 52이면 한 샘플은 52글자 문자열이다.
# 예: ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
cal_len = 52

# 학습 데이터 샘플 수
# 랜덤 문자열 20,000개를 생성해서 학습에 사용한다.
num_samples = 20000

# 난수 고정
# 실행할 때마다 같은 랜덤 데이터가 생성되도록 설정한다.
# 완전히 동일한 결과를 보장하지는 않지만 재현성 확보에 도움된다.
np.random.seed(0)
tf.random.set_seed(0)


# 사용할 문자 집합
# 대문자 26개 + 소문자 26개 = 총 52개 문자
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'

# 실제 문자 개수
num_chars = len(chars)          # 52


# START_TOKEN
# ------------------------------------------------------------
# Teacher Forcing에서 디코더 입력의 첫 번째 위치에 넣는 시작 토큰이다.
#
# 실제 문자는 0~51 인덱스를 사용한다.
# 따라서 START_TOKEN은 실제 문자와 겹치지 않도록 52번을 사용한다.
#
# 예)
#   A → 0
#   B → 1
#   ...
#   z → 51
#   START → 52
# ------------------------------------------------------------
START_TOKEN = num_chars         # 52


# vocab 크기 설정
# ------------------------------------------------------------
# encoder_vocab_size:
#   인코더는 실제 입력 문자만 받는다.
#   따라서 A~Z, a~z 총 52개만 필요하다.
#
# decoder_vocab_size:
#   디코더 입력에는 실제 문자 + START_TOKEN이 들어간다.
#   따라서 52개 문자 + START 1개 = 53개가 필요하다.
#
# output_classes:
#   모델이 최종적으로 예측해야 하는 것은 실제 문자이다.
#   START_TOKEN은 출력 정답에 나오면 안 된다.
#   따라서 출력 클래스 수는 52개이다.
# ------------------------------------------------------------
encoder_vocab_size = num_chars      # 52
decoder_vocab_size = num_chars + 1  # 53
output_classes = num_chars          # 52


# 문자 → 숫자 인덱스 변환 딕셔너리
char_to_index = {char: idx for idx, char in enumerate(chars)}

# 숫자 인덱스 → 문자 변환 딕셔너리
index_to_char = {idx: char for idx, char in enumerate(chars)}


# ------------------------------------------------------------
# 2. 데이터 생성 함수
# ------------------------------------------------------------

def generate_data(num_samples):
    """
    랜덤 문자열 데이터를 생성하는 함수

    X:
        원래 문자열의 문자 인덱스 배열

    y:
        X를 거꾸로 뒤집은 정답 문자열의 문자 인덱스 배열

    예)
        원래 문자열: AbCdE
        X = [0, 27, 2, 29, 4]

        뒤집은 문자열: EdCbA
        y = [4, 29, 2, 27, 0]
    """

    X = []
    y = []

    for _ in range(num_samples):

        # chars에서 cal_len개 문자를 랜덤 선택한다.
        # cal_len = 52이면 52글자 문자열 생성
        sequence = np.random.choice(list(chars), cal_len)

        # 입력 문자열을 문자 인덱스로 변환
        X.append([char_to_index[char] for char in sequence])

        # 정답은 입력 문자열을 뒤집은 값
        # sequence[::-1]은 파이썬에서 순서를 거꾸로 뒤집는 문법이다.
        y.append([char_to_index[char] for char in sequence[::-1]])

    return np.array(X), np.array(y)


# ------------------------------------------------------------
# 3. 데이터 생성
# ------------------------------------------------------------

# X: 인코더에 들어갈 입력 데이터
# y: 디코더가 맞혀야 할 정답 데이터
X, y = generate_data(num_samples)

# 현재 shape 예시
# X.shape = (20000, 52)
# y.shape = (20000, 52)


# ------------------------------------------------------------
# Teacher Forcing용 디코더 입력 생성
# ------------------------------------------------------------
# Seq2Seq에서 디코더는 이전에 나온 값을 보고 다음 값을 예측한다.
#
# 정답 y가 다음과 같다고 하면:
#   y = [E, d, C, b, A]
#
# 디코더 입력은 다음처럼 만든다.
#   decoder_input_data = [START, E, d, C, b]
#
# 즉, 정답 y를 그대로 넣는 것이 아니라
# 한 칸 오른쪽으로 밀어서 넣는다.
#
# 학습 관계:
#   START → E 예측
#   E     → d 예측
#   d     → C 예측
#   C     → b 예측
#   b     → A 예측
# ------------------------------------------------------------

# y와 같은 shape의 배열을 0으로 생성
decoder_input_data = np.zeros_like(y)

# 모든 샘플의 첫 번째 위치에 START_TOKEN 입력
decoder_input_data[:, 0] = START_TOKEN

# 두 번째 위치부터는 y의 마지막 전까지 값을 복사
# y[:, :-1]은 y에서 마지막 글자를 제외한 값
# decoder_input_data[:, 1:]은 첫 번째 칸을 제외한 나머지 칸
decoder_input_data[:, 1:] = y[:, :-1]


# ------------------------------------------------------------
# 4. Seq2Seq + Attention 모델 정의
# ------------------------------------------------------------

# 문자 하나를 몇 차원 벡터로 표현할지 결정
# Embedding Layer가 각 문자 인덱스를 32차원 벡터로 변환한다.
embedding_dim = 32

# LSTM hidden state 크기
# 인코더와 디코더 LSTM의 내부 상태 벡터 크기
hidden_units = 128


# -----------------------------
# 4-1. 인코더
# -----------------------------

# 인코더 입력층
# Embedding을 사용하므로 입력 shape는 (cal_len,)이다.
# 즉, 한 샘플은 52개의 정수 인덱스로 구성된다.
encoder_inputs = Input(shape=(cal_len,), name="encoder_inputs")


# 인코더 Embedding Layer
# ------------------------------------------------------------
# 기존처럼 문자 인덱스를 그대로 LSTM에 넣으면
# A=0, B=1, z=51처럼 숫자 크기 관계가 있는 것처럼 처리될 수 있다.
#
# Embedding을 사용하면 각 문자를 학습 가능한 벡터로 바꾼다.
#
# 입력 shape:
#   (batch_size, cal_len)
#
# 출력 shape:
#   (batch_size, cal_len, embedding_dim)
# ------------------------------------------------------------
encoder_embedding = Embedding(
    input_dim=encoder_vocab_size,
    output_dim=embedding_dim,
    name="encoder_embedding"
)(encoder_inputs)


# 인코더 LSTM
# ------------------------------------------------------------
# return_sequences=True:
#   Attention을 적용하려면 인코더의 마지막 출력만 있으면 안 된다.
#   디코더가 인코더의 모든 시점 출력을 참고해야 하므로
#   전체 시점의 hidden state를 반환해야 한다.
#
# return_state=True:
#   마지막 hidden state(state_h)와 cell state(state_c)를 반환한다.
#   이 두 상태는 디코더 LSTM의 초기 상태로 전달된다.
# ------------------------------------------------------------
encoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)


# encoder_outputs:
#   인코더의 모든 시점 hidden state
#   shape: (batch_size, cal_len, hidden_units)
#
# state_h:
#   인코더 마지막 시점의 hidden state
#
# state_c:
#   인코더 마지막 시점의 cell state
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)


# 디코더 초기 상태로 사용할 인코더 상태
encoder_states = [state_h, state_c]


# -----------------------------
# 4-2. 디코더
# -----------------------------

# 디코더 입력층
# Teacher Forcing으로 만든 decoder_input_data가 들어간다.
decoder_inputs = Input(shape=(cal_len,), name="decoder_inputs")


# 디코더 Embedding Layer
# ------------------------------------------------------------
# 디코더 입력에는 실제 문자 52개뿐 아니라 START_TOKEN도 들어간다.
# 그래서 input_dim은 decoder_vocab_size = 53이다.
# ------------------------------------------------------------
decoder_embedding = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=embedding_dim,
    name="decoder_embedding"
)(decoder_inputs)


# 디코더 LSTM
# ------------------------------------------------------------
# return_sequences=True:
#   디코더는 한 글자만 출력하는 것이 아니라
#   cal_len개의 출력 시퀀스를 만들어야 한다.
#
# return_state=True:
#   디코더의 마지막 상태도 반환한다.
#   현재 학습 모델에서는 반환된 상태를 직접 사용하지 않지만,
#   Seq2Seq 구조상 추론 모델을 분리할 때 자주 사용된다.
# ------------------------------------------------------------
decoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)


# 디코더 실행
# ------------------------------------------------------------
# decoder_embedding:
#   Teacher Forcing용 디코더 입력
#
# initial_state=encoder_states:
#   인코더가 입력 문자열을 읽고 만든 마지막 상태를
#   디코더의 초기 상태로 사용한다.
#
# decoder_outputs:
#   디코더의 모든 시점 hidden state
#   shape: (batch_size, cal_len, hidden_units)
# ------------------------------------------------------------
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)


# -----------------------------
# 4-3. Attention
# -----------------------------

# Attention의 역할
# ------------------------------------------------------------
# 기본 Seq2Seq는 인코더의 마지막 상태만 디코더에 넘긴다.
# 하지만 입력이 길어지면 마지막 상태 하나에 모든 정보를 담기 어렵다.
#
# Attention은 디코더가 각 글자를 출력할 때마다
# 인코더의 전체 시점 출력 중 어느 위치를 더 참고할지 계산한다.
#
# 이 코드에서:
#   query = decoder_outputs
#   key   = encoder_outputs
#   value = encoder_outputs
#
# 즉, 디코더의 현재 상태가 query가 되고,
# 인코더의 각 시점 출력이 key/value가 된다.
# ------------------------------------------------------------
attention_layer = Attention(name="attention_layer")


# context_vectors
# ------------------------------------------------------------
# 디코더의 각 시점마다
# 인코더의 모든 시점 출력에 대한 attention score를 계산하고,
# 그 score를 바탕으로 인코더 출력들을 가중합한 context vector를 만든다.
#
# shape:
#   decoder_outputs: (batch_size, decoder_len, hidden_units)
#   encoder_outputs: (batch_size, encoder_len, hidden_units)
#   context_vectors: (batch_size, decoder_len, hidden_units)
# ------------------------------------------------------------
context_vectors = attention_layer(
    [decoder_outputs, encoder_outputs]
)


# 디코더 출력과 Attention Context 결합
# ------------------------------------------------------------
# decoder_outputs:
#   디코더 LSTM이 만든 현재 출력 정보
#
# context_vectors:
#   Attention이 인코더 전체 출력에서 가져온 참고 정보
#
# 두 벡터를 이어 붙여서 최종 예측에 사용한다.
#
# shape:
#   decoder_outputs   : (batch_size, cal_len, 128)
#   context_vectors   : (batch_size, cal_len, 128)
#   결합 후            : (batch_size, cal_len, 256)
# ------------------------------------------------------------
decoder_combined_context = Concatenate(name="concat_decoder_attention")(
    [decoder_outputs, context_vectors]
)


# -----------------------------
# 4-4. 출력층
# -----------------------------

# 각 시점마다 52개 문자 중 하나를 예측한다.
# softmax는 각 문자에 대한 확률을 출력한다.
decoder_dense = Dense(
    output_classes,
    activation='softmax',
    name="decoder_output_dense"
)


# 최종 출력
# shape:
#   (batch_size, cal_len, output_classes)
#   예: (128, 52, 52)
decoder_outputs_final = decoder_dense(decoder_combined_context)


# ------------------------------------------------------------
# 전체 모델 정의
# ------------------------------------------------------------
# 입력:
#   1. encoder_inputs
#   2. decoder_inputs
#
# 출력:
#   decoder_outputs_final
#
# 학습 시:
#   model.fit([X, decoder_input_data], y)
# ------------------------------------------------------------
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs_final
)


# ------------------------------------------------------------
# 모델 컴파일
# ------------------------------------------------------------
# sparse_categorical_crossentropy:
#   정답 y가 원-핫 인코딩이 아니라 정수 인덱스이므로 사용한다.
#
# 예)
#   정답 A = 0
#   정답 z = 51
#
# metrics=['accuracy']:
#   문자 단위 정확도를 계산한다.
#   문자열 전체가 맞았는지의 정확도는 아래 검증 코드에서 별도로 계산한다.
# ------------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# 모델 구조 출력
model.summary()


# ------------------------------------------------------------
# 5. 모델 학습
# ------------------------------------------------------------

# EarlyStopping
# ------------------------------------------------------------
# val_loss를 관찰하다가 5번 연속 개선되지 않으면 학습을 멈춘다.
#
# restore_best_weights=True:
#   학습이 끝난 뒤 가장 val_loss가 좋았던 시점의 가중치로 복원한다.
#   따라서 fit() 이후 model.save()를 하면 best weight가 저장된다.
# ------------------------------------------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


# 모델 학습
# ------------------------------------------------------------
# 입력:
#   [X, decoder_input_data]
#
# 정답:
#   y
#
# X:
#   인코더 입력
#
# decoder_input_data:
#   Teacher Forcing용 디코더 입력
#
# y:
#   디코더가 최종적으로 맞혀야 하는 정답
# ------------------------------------------------------------
history = model.fit(
    [X, decoder_input_data],
    y,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)


# 모델 저장
# ------------------------------------------------------------
# EarlyStopping에서 restore_best_weights=True를 사용했으므로
# 현재 model은 val_loss 기준 best weight로 복원된 상태이다.
# 따라서 이 시점에 저장하면 best model을 저장하는 효과가 있다.
# ------------------------------------------------------------
model.save("seq2seq_teacher_forcing_attention_best.keras")


# ------------------------------------------------------------
# 6. 예측 함수
# ------------------------------------------------------------

def predict_reverse(input_sequence):
    """
    학습된 모델을 이용해 문자열을 뒤집어 예측하는 함수

    학습 시에는 Teacher Forcing을 사용하므로
    디코더 입력으로 실제 정답의 이전 글자를 넣을 수 있다.

    하지만 예측 시에는 정답을 알 수 없다.

    따라서 예측 시에는:
        1. 첫 번째 디코더 입력에 START_TOKEN을 넣는다.
        2. 첫 번째 글자를 예측한다.
        3. 예측한 글자를 다음 디코더 입력으로 넣는다.
        4. 이 과정을 cal_len번 반복한다.
    """

    # 입력 길이 검사
    # 모델은 cal_len 길이로 학습되었기 때문에
    # 예측할 때도 반드시 같은 길이의 문자열을 입력해야 한다.
    if len(input_sequence) != cal_len:
        raise ValueError(f"입력 문자열은 반드시 {cal_len}글자여야 합니다.")

    # 지원 문자 검사
    # 현재 모델은 A~Z, a~z만 학습했다.
    for char in input_sequence:
        if char not in char_to_index:
            raise ValueError(f"지원하지 않는 문자입니다: {char}")

    # 입력 문자열을 인덱스로 변환
    # 예)
    #   AbCdE → [0, 27, 2, 29, 4]
    input_indices = np.array(
        [char_to_index[char] for char in input_sequence]
    ).reshape((1, cal_len))

    # 디코더 입력 배열 생성
    # 처음에는 전부 0으로 만들고,
    # 첫 번째 위치에 START_TOKEN을 넣는다.
    decoder_input = np.zeros((1, cal_len), dtype=np.int32)
    decoder_input[0, 0] = START_TOKEN

    # 예측된 문자 인덱스를 저장할 리스트
    result_indices = []

    # 한 글자씩 순차적으로 예측
    for t in range(cal_len):

        # 현재까지 채워진 decoder_input을 사용해 전체 시퀀스 예측
        predicted_sequence = model.predict(
            [input_indices, decoder_input],
            verbose=0
        )

        # t번째 위치의 softmax 결과 중 가장 확률이 높은 문자 선택
        pred_idx = np.argmax(predicted_sequence[0, t])

        # 예측 결과 저장
        result_indices.append(pred_idx)

        # 다음 시점의 디코더 입력으로 현재 예측값 사용
        # Teacher Forcing 학습 때는 실제 정답을 넣었지만,
        # 예측 때는 정답을 모르므로 모델이 예측한 값을 넣는다.
        if t + 1 < cal_len:
            decoder_input[0, t + 1] = pred_idx

    # 인덱스를 다시 문자로 변환해서 문자열로 반환
    return ''.join(index_to_char[idx] for idx in result_indices)


# ------------------------------------------------------------
# 7. 검증
# ------------------------------------------------------------

# 테스트용 랜덤 문자열 30개 생성
test_strings = [
    "".join(np.random.choice(list(chars), cal_len))
    for _ in range(30)
]


# 실제 정답 생성
# 각 문자열을 거꾸로 뒤집은 값
expected_outputs = [s[::-1] for s in test_strings]


# 문자열 전체 정확도 계산용 변수
correct_predictions = 0
total_predictions = len(test_strings)


# 테스트 문자열 하나씩 예측
for i, test_string in enumerate(test_strings):

    # 모델 예측 결과
    predicted_output = predict_reverse(test_string)

    # 실제 정답
    expected_output = expected_outputs[i]

    # 문자열 전체가 정확히 일치하는지 확인
    # 한 글자라도 틀리면 False
    is_correct = predicted_output == expected_output

    if is_correct:
        correct_predictions += 1

    print(f"입력: {test_string}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {expected_output}")
    print(f"정확 여부: {'맞음' if is_correct else '틀림'}")
    print("-" * 40)


# 문자열 단위 정확도
# model.fit의 accuracy는 문자 단위 정확도이고,
# 여기서 계산하는 sequence_accuracy는 문자열 전체 정확도이다.
sequence_accuracy = correct_predictions / total_predictions

print(f"문자열 단위 정확도: {sequence_accuracy * 100:.2f}%")
print(f"마지막 val_accuracy는 문자 단위 정확도입니다: {history.history['val_accuracy'][-1]:.4f}")


# ------------------------------------------------------------
# 8. 직접 입력 테스트
# ------------------------------------------------------------

while True:

    # 사용자로부터 직접 문자열 입력
    sample = input(f"\n테스트할 문자열을 입력하세요. 반드시 {cal_len}글자여야 합니다. 종료하려면 q 입력: ")

    # q 입력 시 종료
    if sample.lower() == "q":
        print("테스트를 종료합니다.")
        break

    # 입력 길이 검사
    if len(sample) != cal_len:
        print(f"입력 길이 오류: 현재 {len(sample)}글자입니다. 반드시 {cal_len}글자를 입력해야 합니다.")
        continue

    # 지원하지 않는 문자가 있는지 검사
    invalid_chars = [char for char in sample if char not in char_to_index]

    if invalid_chars:
        print(f"지원하지 않는 문자가 포함되어 있습니다: {invalid_chars}")
        print("사용 가능한 문자는 A~Z, a~z 입니다.")
        continue

    # 모델 예측
    predicted = predict_reverse(sample)

    # 실제 정답
    expected = sample[::-1]

    print("\n직접 테스트 결과")
    print(f"입력: {sample}")
    print(f"예측: {predicted}")
    print(f"정답: {expected}")
    print(f"정확 여부: {'맞음' if predicted == expected else '틀림'}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs (InputLayer)   │ (None, 52)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_inputs (InputLayer)   │ (None, 52)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_embedding (Embedding) │ (None, 52, 32)            │           1,664 │ encoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_embedding (Embedding) │ (None, 52, 32)            │           1,696 │ decoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 52, 128), (None,  │          82,432 │ encoder_embedding[0][0]    │
│                               │ 128), (None, 128)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 52, 128), (None,  │          82,432 │ decoder_embedding[0][0],   │
│                               │ 128), (None, 128)]        │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_layer (Attention)   │ (None, 52, 128)           │               0 │ decoder_lstm[0][0],        │
│                               │                           │                 │ encoder_lstm[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concat_decoder_attention      │ (None, 52, 256)           │               0 │ decoder_lstm[0][0],        │
│ (Concatenate)                 │                           │                 │ attention_layer[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_output_dense (Dense)  │ (None, 52, 52)            │          13,364 │ concat_decoder_attention[… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 181,588 (709.33 KB)

 Trainable params: 181,588 (709.33 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 125ms/step - accuracy: 0.0287 - loss: 3.9337 - val_accuracy: 0.0520 - val_loss: 3.8226
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 108ms/step - accuracy: 0.0628 - loss: 3.7645 - val_accuracy: 0.0967 - val_loss: 3.5830
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 14s 110ms/step - accuracy: 0.1378 - loss: 3.3717 - val_accuracy: 0.3349 - val_loss: 2.3994
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.5003 - loss: 1.8758 